# Pertemuan 10 — Dynamic Programming I
*Overlapping subproblems*, *optimal substructure*, memoization vs tabulation, 0/1 Knapsack, dan kompleksitas pseudo-polinomial.

In [ ]:
import time, random, sys
from functools import lru_cache
import numpy as np
import matplotlib.pyplot as plt
sys.setrecursionlimit(10000); random.seed(10)

## 1. Fibonacci: eksponensial → linear → O(1) memori

In [ ]:
calls = [0]
def fib_naive(n):
    calls[0] += 1
    return n if n < 2 else fib_naive(n - 1) + fib_naive(n - 2)
memo_calls = [0]
@lru_cache(None)
def fib_memo(n):
    memo_calls[0] += 1
    return n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)
def fib_tab(n):
    a, b = 0, 1
    for _ in range(n): a, b = b, a + b
    return a
print(f"{'n':>3} {'panggilan naif':>15} {'panggilan memo':>15}")
for n in [10, 20, 25, 30]:
    calls[0] = 0; memo_calls[0] = 0; fib_memo.cache_clear()
    assert fib_naive(n) == fib_memo(n) == fib_tab(n)
    print(f"{n:>3} {calls[0]:>15} {memo_calls[0]:>15}")
print("fib(1000) tabulasi:", str(fib_tab(1000))[:30], "...")

## 2. 0/1 Knapsack: top-down, bottom-up, dan optimasi ruang

In [ ]:
def knap_topdown(vals, wts, W):
    @lru_cache(None)
    def go(i, c):
        if i == len(vals): return 0
        best = go(i + 1, c)
        if wts[i] <= c: best = max(best, vals[i] + go(i + 1, c - wts[i]))
        return best
    return go(0, W)

def knap_bottomup(vals, wts, W):
    n = len(vals); dp = [[0] * (W + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for c in range(W + 1):
            dp[i][c] = dp[i-1][c]
            if wts[i-1] <= c: dp[i][c] = max(dp[i][c], dp[i-1][c - wts[i-1]] + vals[i-1])
    chosen, c = [], W                                 # rekonstruksi solusi
    for i in range(n, 0, -1):
        if dp[i][c] != dp[i-1][c]: chosen.append(i - 1); c -= wts[i-1]
    return dp[n][W], sorted(chosen)

def knap_1d(vals, wts, W):                              # ruang O(W), iterasi kapasitas turun
    dp = [0] * (W + 1)
    for v, w in zip(vals, wts):
        for c in range(W, w - 1, -1): dp[c] = max(dp[c], dp[c - w] + v)
    return dp[W]

vals, wts, W = [60, 100, 120, 30, 75], [10, 20, 30, 5, 15], 50
print(knap_topdown(vals, wts, W), knap_bottomup(vals, wts, W), knap_1d(vals, wts, W))

## 3. Brute force O(2ⁿ) vs DP O(nW)

In [ ]:
import itertools
def knap_brute(vals, wts, W):
    best = 0
    for mask in range(1 << len(vals)):
        tw = sum(wts[i] for i in range(len(vals)) if mask >> i & 1)
        if tw <= W: best = max(best, sum(vals[i] for i in range(len(vals)) if mask >> i & 1))
    return best
print(f"{'n':>3} {'brute(s)':>10} {'DP(s)':>10}")
for n in [10, 14, 18, 20]:
    v = [random.randint(1, 100) for _ in range(n)]; w = [random.randint(1, 30) for _ in range(n)]; Wn = sum(w) // 2
    t = time.perf_counter(); a = knap_brute(v, w, Wn); t1 = time.perf_counter() - t
    t = time.perf_counter(); b = knap_1d(v, w, Wn);    t2 = time.perf_counter() - t
    assert a == b; print(f"{n:>3} {t1:>10.4f} {t2:>10.5f}")

## 4. Pseudo-polinomial: waktu ∝ W, padahal W hanya butuh log W bit

In [ ]:
n = 50; v = [random.randint(1, 100) for _ in range(n)]; w = [random.randint(1, 1000) for _ in range(n)]
Ws, ts = [10**3, 10**4, 10**5], []
for Wc in Ws:
    t = time.perf_counter(); knap_1d(v, w, Wc); ts.append(time.perf_counter() - t)
    print(f"W={Wc:>7}  ({Wc.bit_length()} bit)  waktu={ts[-1]:.3f}s")
print("Ketika W naik 10×, waktu naik ≈", round(ts[1] / ts[0], 1), "× dan", round(ts[2] / ts[1], 1), "× → linear terhadap NILAI W, eksponensial terhadap panjang inputnya.")

## 5. Memori: tabel 2D vs 1D

In [ ]:
for n, Wc in [(100, 10**4), (1000, 10**5)]:
    print(f"n={n}, W={Wc}: tabel 2D = {(n+1)*(Wc+1)*8/1e6:.1f} MB (int64) vs 1D = {(Wc+1)*8/1e6:.2f} MB")

## 6. Latihan / Tugas
1. Tulis *recurrence* dan analisis waktu/ruang untuk **Longest Increasing Subsequence** (O(n²) DP, lalu O(n log n) dengan `bisect`).
2. Ubah knapsack menjadi **unbounded knapsack** (arah iterasi kapasitas berubah — mengapa?).
3. **Subset sum** dengan `bitset` Python (integer sebagai bit) — ukur percepatan terhadap DP list.
4. **Tugas:** tabel waktu/ruang untuk top-down vs bottom-up vs 1D pada beberapa (n, W); jelaskan overhead rekursi dan batas kedalaman rekursi Python.

In [ ]:
def lis(a):
    pass  # TODO